# 1. Imports and Data Loading


In [1]:
# Install packages and NLTK sentence tokenizer resources.
!pip install -q ftfy beautifulsoup4 nltk spacy sentence-transformers bertopic umap-learn hdbscan scikit-learn langdetect gensim
!python -m spacy download en_core_web_sm

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 12.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 55.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 111.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
# Import libraries.
import re
import numpy as np
import pandas as pd

import ftfy
from bs4 import BeautifulSoup

import nltk
from nltk.tokenize import sent_tokenize

import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

from sentence_transformers import SentenceTransformer

from bertopic import BERTopic
from umap import UMAP
import hdbscan
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/airbnb_review_text_analytics')

from langdetect import detect

Mounted at /content/drive


In [ ]:
# Load the raw Airbnb review data and remove reviews without comments.
reviews = pd.read_csv('data/raw/reviews.csv.gz', compression='gzip', low_memory=False)

reviews = reviews.dropna(subset=["comments"]).reset_index(drop=True)

print("Total reviews:", len(reviews))
print(reviews.head())

Total reviews: 651456
   listing_id         id        date  reviewer_id reviewer_name  \
0     9974111  105972997  2016-10-03     74630712          Lily   
1     9974111  125205050  2017-01-04        23609          Mark   
2     9974111  191232367  2017-09-06    147270253        Dianne   
3     9974111  196001139  2017-09-21      5590969        Charie   
4     9974111  197975881  2017-09-27     76030814        Oswald   

                                            comments  
0  I absolutely loved the location!! I was there ...  
1  Stayed here for 2 weeks , great location , gre...  
2  Awesome location! Communication with Shoug was...  
3  Had a really great two week stay at Shoug's pl...  
4             great and easy stay - no issues at all  


In [ ]:
# Use a reproducible review sample to make BERTopic training faster during development.
MAX_REVIEWS = 10000  # Change to None to use the full review dataset.
if MAX_REVIEWS is not None and len(reviews) > MAX_REVIEWS:
    reviews = reviews.sample(MAX_REVIEWS, random_state=42).reset_index(drop=True)

print("Reviews used:", len(reviews))

Reviews used: 10000


# 2. Review Segmentation


In [ ]:
# Split reviews into smaller text segments before topic modeling.
# Airbnb reviews often contain multiple topics and mixed sentiments in one sentence.
# Therefore, after sentence tokenization, sentences are further split by commas,
# semicolons, and contrastive connectors such as "but", "however", "although", and "while".
MIN_WORDS_SEGMENT = 2


def basic_clean(text: str) -> str:
    """
    Remove HTML tags, control characters, and extra spaces from one review.
    """
    text = str(text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = re.sub(r"[\x00-\x1F\x7F]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_review_text(text: str) -> list:
    """
    Split one review into topic-focused segments for more precise topic modeling.
    """
    if pd.isna(text):
        return []

    text = basic_clean(text)
    sentences = sent_tokenize(text)

    segments = []

    for sentence in sentences:
        sentence = sentence.strip()

        if len(sentence.split()) < MIN_WORDS_SEGMENT:
            continue

        parts = re.split(
            r",|;|\bbut\b|\bhowever\b|\balthough\b|\bwhile\b",
            sentence,
            flags=re.IGNORECASE
        )

        parts = [p.strip() for p in parts if p.strip()]

        if len(parts) > 1:
            segments.extend(parts)
        else:
            segments.append(sentence)

    segments = [
        s for s in segments
        if len(s.split()) >= MIN_WORDS_SEGMENT
    ]

    return segments


rows = []

for _, r in reviews.iterrows():
    review_id = r["id"]
    listing_id = r["listing_id"]

    segments = tokenize_review_text(r["comments"])

    for segment in segments:
        rows.append({
            "review_id": review_id,
            "listing_id": listing_id,
            "segment_text": segment
        })


segment_df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)

print("Segments:", len(segment_df))
segment_df.head()

Segments: 47473


,review_id,listing_id,segment_text
0,1389200671170187783,1072485313113718664,Soggiorno splendido e zona di alloggio molto c...
1,1389200671170187783,1072485313113718664,Zone di interesse facili da raggiungere sia a ...
2,463567767,10412695,This place was great!
3,463567767,10412695,The pub downstairs has awesome food
4,463567767,10412695,and seriously the best ceasar I've ever had.


# 3. Language Filtering


In [ ]:
# Detect the language of each segment so that non-English segments can be removed.
def detect_language(text):
    try:
        return detect(text)
    except:
        return "unknown"

segment_df['language'] = segment_df['segment_text'].apply(detect_language)
segment_df['language'].value_counts()

,count
language,
en,40744
fr,2094
es,910
de,436
pt,427
af,379
ro,368
it,337
ca,207


In [ ]:
# Keep English segments only for a consistent English topic model.
segment_df = segment_df[segment_df['language'] == 'en']
segment_df = segment_df.drop('language', axis=1)

print("Segments after non-English removal:", len(segment_df))

Segments after non-English removal: 40744


# 4. Text Normalization


In [ ]:
# Normalize each segment before embedding and topic modeling.
# Stop words are removed, but negation words are preserved because they can change sentiment meaning.
NEGATION_WORDS = {"not", "no", "never"}
MIN_WORDS_SEG_NORM = 2

_punct_re = re.compile(r"[^\w\s]")


def normalize_segment(segment: str) -> str:
    segment = segment.lower()
    segment = _punct_re.sub(" ", segment)
    segment = re.sub(r"\s+", " ", segment).strip()

    doc = nlp(segment)

    tokens = []

    for tok in doc:
        if tok.is_space:
            continue

        if tok.is_stop and tok.text not in NEGATION_WORDS:
            continue

        if tok.is_punct:
            continue

        lemma = tok.lemma_.strip()

        if len(lemma) < 2:
            continue

        tokens.append(lemma)

    if len(tokens) < MIN_WORDS_SEG_NORM:
        return ""

    return " ".join(tokens)


segment_df["segment_norm"] = segment_df["segment_text"].apply(normalize_segment)
segment_df = segment_df[segment_df["segment_norm"] != ""].reset_index(drop=True)

print("Segments after normalization:", len(segment_df))
segment_df.head()

Segments after normalization filter: 38145


,segment_id,review_id,listing_id,segment_text,segment_norm
0,0,463567767,10412695,This place was great!,place great
1,1,463567767,10412695,The pub downstairs has awesome food,pub downstairs awesome food
2,2,463567767,10412695,and seriously the best ceasar I've ever had.,seriously good ceasar ve
3,3,463567767,10412695,The roof top patio was amazing,roof patio amazing
4,4,463567767,10412695,and a great spot to chill out and people watch.,great spot chill people watch


In [ ]:
# Save the cleaned and normalized segment-level dataset for reproducibility.
os.makedirs("data/processed", exist_ok=True)
segment_df.to_csv("data/processed/segment_df.csv", index=False)

# 5. Sentence Embedding Generation


In [ ]:
# Encode normalized segments with a sentence-transformer model.
embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(
    segment_df["segment_norm"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embeddings shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/299 [00:00<?, ?it/s]

Embeddings shape: (38145, 384)


# 6. BERTopic Model Training


In [ ]:
# Train the BERTopic model using UMAP for dimensionality reduction and HDBSCAN for clustering.
TOPIC_MIN_CLUSTER_SIZE = 200
TOPIC_MIN_SAMPLES = 30

umap_model = UMAP(
    n_neighbors=20,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=TOPIC_MIN_CLUSTER_SIZE,
    min_samples=TOPIC_MIN_SAMPLES,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    stop_words="english",
    min_df=30
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(segment_df["segment_norm"].tolist(), embeddings)

segment_df["topic_id"] = topics
segment_df["is_outlier"] = (segment_df["topic_id"] == -1).astype(int)
segment_df["topic_prob"] = [
    float(np.max(p)) if p is not None else np.nan
    for p in probs
]

2026-04-30 15:22:27,830 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-30 15:23:56,656 - BERTopic - Dimensionality - Completed ✓
2026-04-30 15:23:56,658 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-30 15:24:12,546 - BERTopic - Cluster - Completed ✓
2026-04-30 15:24:12,558 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-30 15:24:12,841 - BERTopic - Representation - Completed ✓


In [ ]:
# Save the segment-level topic assignments produced by BERTopic.
segment_df.to_csv("data/processed/segment_df_with_topics.csv", index=False)
print("Saved: segment_df_with_topics.csv")

Saved: segment_df_with_topics.csv


# 7. BERTopic Evaluation


In [ ]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Calculate the outlier rate and the number of non-outlier topics.
n_total_segments = len(segment_df)
n_outlier_segments = (segment_df["topic_id"] == -1).sum()
outlier_rate = n_outlier_segments / n_total_segments

n_topics = segment_df.loc[
    segment_df["topic_id"] != -1,
    "topic_id"
].nunique()

print("Total segments:", n_total_segments)
print("Number of topics:", n_topics)
print("Outlier segments:", n_outlier_segments)
print("Outlier rate:", round(outlier_rate, 4))

# Calculate topic diversity at 10.
# This metric checks how much the top words differ across topics.
# A higher value means less keyword overlap across topics.
def compute_topic_diversity(topic_model, top_n=10):
    topic_words_all = []

    for topic_id, words in topic_model.get_topics().items():
        if topic_id == -1:
            continue

        top_words = [word for word, _ in words[:top_n]]
        topic_words_all.extend(top_words)

    if len(topic_words_all) == 0:
        return np.nan

    return len(set(topic_words_all)) / len(topic_words_all)


topic_diversity = compute_topic_diversity(topic_model, top_n=10)

print("\nTopic diversity @10:", round(topic_diversity, 4))

Total segments: 38145
Number of topics: 53
Outlier segments: 8877
Outlier rate: 0.2327

Topic diversity @10: 0.2245


In [ ]:
# Calculate c_v topic coherence.
# This metric checks whether the top words within each topic are semantically coherent.
# A higher value suggests stronger interpretability within topics.
texts_tokenized = (
    segment_df["segment_norm"]
    .dropna()
    .astype(str)
    .apply(lambda x: x.split())
    .tolist()
)

dictionary = Dictionary(texts_tokenized)

# Remove extremely rare and overly common words before coherence calculation.
dictionary.filter_extremes(no_below=5, no_above=0.5)

topic_words = []

for topic_id, words in topic_model.get_topics().items():
    if topic_id == -1:
        continue

    top_words = [word for word, _ in words[:10]]

    # Keep only words that exist in the filtered dictionary.
    top_words = [word for word in top_words if word in dictionary.token2id]

    if len(top_words) > 0:
        topic_words.append(top_words)

coherence_model_cv = CoherenceModel(
    topics=topic_words,
    texts=texts_tokenized,
    dictionary=dictionary,
    coherence="c_v"
)

topic_coherence_cv = coherence_model_cv.get_coherence()

print("Topic coherence c_v:", round(topic_coherence_cv, 4))

Topic coherence c_v: 0.3957


In [ ]:
# Calculate embedding-based clustering metrics for the original BERTopic assignments.
# Outliers are excluded because they are not assigned to stable topic clusters.
labels = np.array(segment_df["topic_id"])
emb = np.array(embeddings)

valid_mask = labels != -1

X_valid = emb[valid_mask]
labels_valid = labels[valid_mask]

print("Valid clustered segments:", len(labels_valid))
print("Valid topics:", len(set(labels_valid)))

# Use sampling because silhouette score can be slow on a large segment-level dataset.
sample_size = min(10000, len(labels_valid))

np.random.seed(42)
sample_idx = np.random.choice(
    len(labels_valid),
    size=sample_size,
    replace=False
)

X_sample = X_valid[sample_idx]
labels_sample = labels_valid[sample_idx]

silhouette = silhouette_score(
    X_sample,
    labels_sample,
    metric="cosine"
)

davies_bouldin = davies_bouldin_score(
    X_sample,
    labels_sample
)

print("Silhouette Score:", round(silhouette, 4))
print("Davies-Bouldin Index:", round(davies_bouldin, 4))

Valid clustered segments: 29268
Valid topics: 53
Silhouette Score: 0.0728
Davies-Bouldin Index: 2.9999


In [ ]:
# Create a compact evaluation summary table for the original BERTopic model.
# This table is used to summarize model quality before manual topic mapping.
bertopic_evaluation_summary = pd.DataFrame({
    "metric": [
        "outlier_rate",
        "number_of_topics",
        "topic_diversity_at_10",
        "topic_coherence_c_v",
        "silhouette_score",
        "davies_bouldin_index"
    ],
    "value": [
        outlier_rate,
        n_topics,
        topic_diversity,
        topic_coherence_cv,
        silhouette,
        davies_bouldin
    ]
})

display(bertopic_evaluation_summary)

,metric,value
0,outlier_rate,0.232717
1,number_of_topics,53.000000
2,topic_diversity_at_10,0.224528
3,topic_coherence_c_v,0.395709
4,silhouette_score,0.072805
5,davies_bouldin_index,2.999943


Saved: data/processed/bertopic_evaluation_summary.csv


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Build a topic dictionary table to make the raw BERTopic results interpretable.
topic_info = topic_model.get_topic_info()
rep_docs = topic_model.get_representative_docs()

TOPIC_PHRASES_K = 10

def get_top_phrases_for_topic(docs, top_k=10):
    """
    Return the most frequent 2-word and 3-word phrases within one topic.
    """
    if len(docs) == 0:
        return []

    vectorizer = CountVectorizer(
        ngram_range=(2, 3),
        stop_words="english",
        min_df=2
    )

    try:
        phrase_matrix = vectorizer.fit_transform(docs)
    except ValueError:
        return []

    phrase_counts = phrase_matrix.sum(axis=0).A1
    phrases = vectorizer.get_feature_names_out()

    phrase_freq = pd.DataFrame({
        "phrase": phrases,
        "count": phrase_counts
    }).sort_values("count", ascending=False)

    return phrase_freq["phrase"].head(top_k).tolist()


topic_dict_rows = []

for _, r in topic_info.iterrows():
    tid = int(r["Topic"])

    if tid == -1:
        continue

    topic_docs = segment_df.loc[
        segment_df["topic_id"] == tid,
        "segment_norm"
    ].dropna().astype(str).tolist()

    top_phrases = get_top_phrases_for_topic(
        topic_docs,
        top_k=TOPIC_PHRASES_K
    )

    reps = rep_docs.get(tid, [])[:3]

    topic_dict_rows.append({
        "topic_id": tid,
        "topic_label": f"Topic_{tid}",
        "top_phrases": ", ".join(top_phrases),
        "n_segments_topic": int(r["Count"]),
        "representative_segments": " || ".join(reps)
    })


topic_dictionary_table = pd.DataFrame(topic_dict_rows).sort_values(
    "n_segments_topic",
    ascending=False
).reset_index(drop=True)

topic_dictionary_table.head()

,topic_id,topic_label,top_phrases,n_segments_topic,representative_segments
0,0,Topic_0,"great host, amazing host, wonderful host, frie...",3046,host respond quickly || s hard hold host accou...
1,1,Topic_1,"stay toronto, visit toronto, downtown toronto,...",1928,definitely stay visit toronto || definitely st...
2,2,Topic_2,"grocery store, great restaurant, walk distance...",1347,close lot restaurant || close public transport...
3,3,Topic_3,"great space, nicely decorate, beautifully deco...",1265,gorgeous space || wonderfully decorate space c...
4,4,Topic_4,"place clean, clean place, room clean, clean co...",1063,bedroom clean || clean neat || immaculately clean


In [4]:
# Save the raw topic dictionary before manual topic labeling.
bertopic_evaluation_summary.to_csv("data/processed/bertopic_evaluation_summary.csv", index=False)
topic_dictionary_table.to_csv("data/processed/topic_dictionary_table.csv", index=False)
print("Saved: bertopic_evaluation_summary.csv and topic_dictionary_table.csv")

Saved: bertopic_evaluation_summary.csv and topic_dictionary_table.csv


In 2_neuron_network_and_lda.ipynb, we also implement Neural Autoencoder + K-Means and LDA as comparison methods to evaluate topic modeling performance based on quantitative metrics and topic interpretability.

Although these methods achieve stronger scores on some metrics, BERTopic is selected as the final method because its topics are more interpretable and better aligned with Airbnb review themes.

# 8. Manual Topic Mapping


In [ ]:
# Map raw BERTopic topics to human-readable labels.
# The original model generated 53 detailed topics, and some of them are semantically similar.
# This manual mapping groups similar topics under clearer topic labels and adds a broader main topic label.
# The main topic label provides a more macro-level summary for reporting and attention analysis.
new_topic_label_mapping = {
    0:  ("Communication", "Host Quality"),
    1:  ("Experience", "Overall Experience"),
    2:  ("Location", "Nearby Restaurants & Shops"),
    3:  ("Comfort", "Interior Design"),
    4:  ("Cleanliness", "General Cleanliness"),
    5:  ("Experience", "Recommendation"),
    6:  ("Check-in", "Check-in"),
    7:  ("Location", "Neighborhood Quietness"),
    8:  ("Location", "Location Convenience"),
    9:  ("Accuracy", "Apartment Expectation"),
    10: ("Location", "Transportation Access"),
    11: ("Experience", "Home Feeling"),
    12: ("Experience", "Overall Experience"),
    13: ("Comfort", "Bed Comfort"),
    14: ("Experience", "Overall Experience"),
    15: ("Amenities", "Kitchen Facilities"),
    16: ("Amenities", "Parking"),
    17: ("Location", "Location Convenience"),
    18: ("Experience", "Recommendation"),
    19: ("Communication", "Communication"),
    20: ("Comfort", "Interior Space & Natural Light"),
    21: ("Cleanliness", "Bathroom Cleanliness"),
    22: ("Communication", "Responsive Host"),
    23: ("Communication", "Responsive Host"),
    24: ("Location", "Location Convenience"),
    25: ("Communication", "Quick Response"),
    26: ("Experience", "Overall Experience"),
    27: ("Communication", "Issue Handling"),
    28: ("Location", "Location Convenience"),
    29: ("Experience", "View Experience"),
    30: ("Location", "Transportation Access"),
    31: ("Accuracy", "Photo Accuracy"),
    32: ("Experience", "Overall Experience"),
    33: ("Amenities", "Laundry & Towels"),
    34: ("Value", "Value"),
    35: ("Experience", "View Experience"),
    36: ("Experience", "Overall Experience"),
    37: ("Location", "Access & Entry"),
    38: ("Location", "Location Convenience"),
    39: ("Amenities", "Coffee & Breakfast"),
    40: ("Experience", "Recommendation"),
    41: ("Comfort", "Overall Comfort"),
    42: ("Cleanliness", "General Cleanliness"),
    43: ("Cleanliness", "General Cleanliness"),
    44: ("Amenities", "Temperature Control"),
    45: ("Amenities", "General Amenities"),
    46: ("Comfort", "Overall Comfort"),
    47: ("Experience", "Overall Experience"),
    48: ("Comfort", "Basement & Ceiling"),
    49: ("Location", "Tourist & Event Attractions"),
    50: ("Experience", "Overall Experience"),
    51: ("Amenities", "Wi-Fi & TV"),
    52: ("Experience", "Overall Experience"),
}

topic_dictionary_table["main_topic_label"] = topic_dictionary_table["topic_id"].map(
    lambda x: new_topic_label_mapping[x][0]
)

topic_dictionary_table["topic_label"] = topic_dictionary_table["topic_id"].map(
    lambda x: new_topic_label_mapping[x][1]
)

topic_dictionary_table.head()

,topic_id,topic_label,top_phrases,n_segments_topic,representative_segments,main_topic_label
0,0,Host Quality,"great host, amazing host, wonderful host, frie...",3046,host respond quickly || s hard hold host accou...,Communication
1,1,Overall Experience,"stay toronto, visit toronto, downtown toronto,...",1928,definitely stay visit toronto || definitely st...,Experience
2,2,Nearby Restaurants & Shops,"grocery store, great restaurant, walk distance...",1347,close lot restaurant || close public transport...,Location
3,3,Interior Design,"great space, nicely decorate, beautifully deco...",1265,gorgeous space || wonderfully decorate space c...,Comfort
4,4,General Cleanliness,"place clean, clean place, room clean, clean co...",1063,bedroom clean || clean neat || immaculately clean,Cleanliness


In [ ]:
# Reset topic IDs after manual mapping.
# Topics with the same main_topic_label and topic_label are assigned the same new_topic_id.
topic_dictionary_table_final = topic_dictionary_table.copy()

topic_dictionary_table_final = topic_dictionary_table_final.rename(
    columns={"topic_id": "original_topic_id"}
)

main_topic_mapping = {
    label: idx
    for idx, label in enumerate(
        sorted(topic_dictionary_table_final["main_topic_label"].dropna().unique())
    )
}

topic_dictionary_table_final["main_topic_id"] = topic_dictionary_table_final["main_topic_label"].map(main_topic_mapping)

topic_dictionary_table_final = topic_dictionary_table_final.sort_values(
    ["main_topic_id", "topic_label", "original_topic_id"]
).reset_index(drop=True)

topic_dictionary_table_final["new_topic_id"] = (
    topic_dictionary_table_final
    .groupby(["main_topic_id", "topic_label"], sort=False)
    .ngroup()
)

topic_dictionary_table_final = topic_dictionary_table_final[
    [
        "main_topic_id",
        "main_topic_label",
        "original_topic_id",
        "new_topic_id",
        "topic_label",
        "top_phrases",
        "n_segments_topic",
        "representative_segments"
    ]
]

topic_dictionary_table_final.head()

,main_topic_id,main_topic_label,original_topic_id,new_topic_id,topic_label,top_phrases,n_segments_topic,representative_segments
0,0,Accuracy,9,0,Apartment Expectation,"apartment great, great apartment, apartment pe...",707,apartment equiped || no fault apartment owner ...
1,0,Accuracy,31,1,Photo Accuracy,"exactly like, like picture, exactly picture, p...",331,place look exactly like picture || place look ...
2,1,Amenities,39,2,Coffee & Breakfast,"breakfast morning, coffee tea, coffee machine,...",286,local recommendation snack provide || provide ...
3,1,Amenities,45,3,General Amenities,"amenity need, great amenity, lot amenity, clea...",250,cosy require amenity || fabulous property full...
4,1,Amenities,15,4,Kitchen Facilities,"kitchen stock, fully equip, kitchen equipped, ...",592,small oven use || include food fridge didn t u...


In [ ]:
# Save the final topic dictionary after manual mapping and topic ID reset.
topic_dictionary_table_final.to_csv("data/processed/topic_dictionary_table_final.csv", index=False)
print("Saved: topic_dictionary_table_final.csv")

Saved: topic_dictionary_table_final.csv


In [ ]:
# Merge the final topic IDs and labels back to the segment-level dataset.
# Outlier segments are removed because they do not represent stable topic assignments.
topic_mapping = topic_dictionary_table_final.iloc[:, :5].copy()

segment_df_final = segment_df.merge(
    topic_mapping,
    left_on="topic_id",
    right_on="original_topic_id",
    how="left"
)

segment_df_final = segment_df_final[
    segment_df_final["topic_id"] != -1
]

# Remove original BERTopic columns that are no longer needed in the final segment table.
segment_df_final = segment_df_final.drop(
    columns=["topic_id", "is_outlier", "topic_prob", "original_topic_id"]
)

segment_df_final = segment_df_final.rename(
    columns={"new_topic_id": "topic_id"}
)

segment_df_final["main_topic_id"] = segment_df_final["main_topic_id"].astype(int)
segment_df_final["topic_id"] = segment_df_final["topic_id"].astype(int)

print(segment_df_final.shape)
segment_df_final.head()

(29268, 9)


,segment_id,review_id,listing_id,segment_text,segment_norm,main_topic_id,main_topic_label,topic_id,topic_label
1,1,463567767,10412695,The pub downstairs has awesome food,pub downstairs awesome food,7,Location,28,Nearby Restaurants & Shops
2,2,463567767,10412695,and seriously the best ceasar I've ever had.,seriously good ceasar ve,6,Experience,23,Overall Experience
3,3,463567767,10412695,The roof top patio was amazing,roof patio amazing,6,Experience,25,View Experience
5,5,463567767,10412695,The neighborhood is perfect,neighborhood perfect,7,Location,27,Location Convenience
9,9,1515313053468489649,1467062721079452795,Came back to the city to visit friends,come city visit friend,7,Location,27,Location Convenience


In [ ]:
# Save the final segment-level dataset with main topic and final topic IDs.
segment_df_final.to_csv("data/processed/segment_df_with_topics_final.csv", index=False)
print("Saved: segment_df_with_topics_final.csv")

Saved: segment_df_with_topics_final.csv


# 9. Post-Mapping Metric Check


In [ ]:
# Recalculate selected clustering metrics after manual topic mapping and topic ID reset.
# Outliers are excluded using the same rule as the original clustering evaluation for a consistent comparison.
number_of_topics = segment_df_final["topic_id"].nunique()

emb = np.array(embeddings)
valid_mask = np.array(segment_df["topic_id"]) != -1

X_valid = emb[valid_mask]
labels_valid = np.array(segment_df_final["topic_id"])

# Use sampling for consistency with the original clustering metric calculation.
sample_size = min(10000, len(labels_valid))

np.random.seed(42)
sample_idx = np.random.choice(
    len(labels_valid),
    size=sample_size,
    replace=False
)

X_sample = X_valid[sample_idx]
labels_sample = labels_valid[sample_idx]

silhouette = silhouette_score(
    X_sample,
    labels_sample,
    metric="cosine"
)

davies_bouldin = davies_bouldin_score(
    X_sample,
    labels_sample
)

print("Number of Topics:", number_of_topics)
print("Silhouette Score:", round(silhouette, 4))
print("Davies-Bouldin Index:", round(davies_bouldin, 4))

Number of Topics: 33
Silhouette Score: 0.0785
Davies-Bouldin Index: 3.2792


After merging similar interpreted topics, the number of topics decreased from 53 to 33, making the final topic structure more concise and easier to interpret.

The silhouette score slightly increased from 0.0728 to 0.0785, suggesting a small improvement in topic separation after grouping similar topics.

However, the Davies-Bouldin index increased from 2.9999 to 3.2792, meaning that the merged topics became less compact internally.

Therefore, the mapping improves interpretability and simplicity rather than overall clustering performance.

In [ ]:
# Create attention tables for macro-level and detailed topics.
main_topic_attention = (
    segment_df_final
    .groupby(["main_topic_id", "main_topic_label"])
    .size()
    .reset_index(name="n_segments_main_topic")
)

main_topic_attention["attention_main_topic"] = (
    main_topic_attention["n_segments_main_topic"] / len(segment_df_final)
)

main_topic_attention_table = (
    main_topic_attention[
        ["main_topic_id", "main_topic_label", "attention_main_topic", "n_segments_main_topic"]
    ]
    .sort_values("attention_main_topic", ascending=False)
    .reset_index(drop=True)
)

# Count detailed topics within each main topic.
topic_counts = (
    segment_df_final
    .groupby(["main_topic_id", "main_topic_label", "topic_id", "topic_label"])
    .size()
    .reset_index(name="n_segments_topic")
)

main_topic_size_map = dict(
    main_topic_attention[["main_topic_id", "n_segments_main_topic"]].values
)

topic_counts["attention_topic_within_main_topic"] = topic_counts.apply(
    lambda r: r["n_segments_topic"] / main_topic_size_map.get(r["main_topic_id"], 1),
    axis=1
)

topic_counts["attention_topic_overall"] = (
    topic_counts["n_segments_topic"] / len(segment_df_final)
)

topic_attention_table = (
    topic_counts[
        [
            "main_topic_id",
            "main_topic_label",
            "topic_id",
            "topic_label",
            "attention_topic_within_main_topic",
            "attention_topic_overall",
            "n_segments_topic"
        ]
    ]
    .sort_values(
        ["main_topic_id", "attention_topic_within_main_topic"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)

print("Main Topic Attention Table")
display(main_topic_attention_table)

print("Topic Attention Table")
display(topic_attention_table)

Main Topic Attention Table


,main_topic_id,main_topic_label,attention_main_topic,n_segments_main_topic
0,6,Experience,0.273097,7993
1,7,Location,0.208009,6088
2,5,Communication,0.177566,5197
3,4,Comfort,0.108207,3167
4,1,Amenities,0.085452,2501
5,3,Cleanliness,0.071272,2086
6,0,Accuracy,0.035465,1038
7,2,Check-in,0.029964,877
8,8,Value,0.010968,321


Topic Attention Table


,main_topic_id,main_topic_label,topic_id,topic_label,attention_topic_within_main_topic,attention_topic_overall,n_segments_topic
0,0,Accuracy,0,Apartment Expectation,0.681118,0.024156,707
1,0,Accuracy,1,Photo Accuracy,0.318882,0.011309,331
2,1,Amenities,4,Kitchen Facilities,0.236705,0.020227,592
3,1,Amenities,6,Parking,0.232307,0.019851,581
4,1,Amenities,5,Laundry & Towels,0.129948,0.011104,325
5,1,Amenities,2,Coffee & Breakfast,0.114354,0.009772,286
6,1,Amenities,7,Temperature Control,0.104758,0.008952,262
7,1,Amenities,3,General Amenities,0.099960,0.008542,250
8,1,Amenities,8,Wi-Fi & TV,0.081967,0.007004,205
9,2,Check-in,9,Check-in,1.000000,0.029964,877
